In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

ERROR: Error in parse(text = input): <text>:5:8: unexpected symbol
4: 
5: import numpy
          ^


In [6]:
list.files("/kaggle/input", recursive = TRUE)

character(0)

In [39]:
library(xgboost)
library(caret)
library(Matrix)
test_data <- read.csv("/kaggle/input/datasets/zksdawn4pm/regression-dataset/regression_test.csv")
train_data <- read.csv("/kaggle/input/datasets/zksdawn4pm/regression-dataset/regression_train.csv")

In [35]:
# add_feats <- function(df){
#   pos <- c("alwaysCalm","alwaysHaveFun","alwaysLoveAndCareForYourself","iFindMostThingsAmusing",
#            "iAmWellSatisfiedAboutEverythingInMyLife","iFindBeautyInSomeThings","iAlwaysHaveACheerfulEffectOnOthers",
#            "iUsuallyHaveAGoodInfluenceOnEvents","alwaysMakingProgress","iFeelThatLifeIsVeryRewarding",
#            "iHaveVeryWarmFeelingsTowardsAlmostEveryone","lifeIsGood","iLaughALot")
#   neg <- c("alwaysStressed","alwaysSerious","alwaysDepressed","iRarelyWakeUpFeelingRested",
#            "iDoNotThinkThatTheWorldIsAGoodPlace","iFeelThatIAmNotEspeciallyInControlOfMyLife",
#            "iDontHaveFunWithOtherPeople","iDontFeelParticularlyPleasedWithTheWayIAm",
#            "iAmNotParticularlyOptimisticAboutTheFuture","iDontThinkILookAttractive")
#   df$pos_score <- rowMeans(df[, intersect(pos,names(df))])
#   df$neg_score <- rowMeans(df[, intersect(neg,names(df))])
#   df$balance   <- df$pos_score - df$neg_score
#   inc <- c("0 - 10k","10k - 15k","15k - 20k","20k - 50k","50k - 80k","80k - 120k","120k - 150k","150k - 200k","200k above")
#   df$income_ord <- match(df$income, inc) - 1
#   df$inc_x_stress  <- df$income_ord * df$alwaysStressed
#   df$inc_x_balance <- df$income_ord * df$balance
#   df$inc_x_pos     <- df$income_ord * df$pos_score
#   df$inc_x_neg     <- df$income_ord * df$neg_score
#   df
# }
# train_data <- add_feats(train_data)    # 在去happiness、rbind、dummyVars之前
# test_data  <- add_feats(test_data)
# # 然后照常: y<-...; combined<-rbind(...); dummyVars(...) （income仍当factor做one-hot, income_ord等数值列原样保留）

In [40]:

y <- train_data$happiness
train_data$happiness <- NULL

# 在合并数据上做一致的one-hot（参考那份的好写法）
combined <- rbind(train_data, test_data)
dummies  <- dummyVars(" ~ .", data = combined)
enc      <- predict(dummies, newdata = combined)
X     <- as.matrix(enc[1:nrow(train_data), ])
Xtest <- as.matrix(enc[(nrow(train_data)+1):nrow(combined), ])

dtrain <- xgb.DMatrix(data = X, label = y)
dtest  <- xgb.DMatrix(data = Xtest)

# # 配置A（推荐：强正则，防止背噪声）
# params <- list(objective="reg:squarederror",
#                eta=0.03, max_depth=3, min_child_weight=5,
#                subsample=0.7, colsample_bytree=0.7,
#                gamma=0, lambda=1)

# 用xgb.cv + early stopping自动定nrounds（防止极端过拟合）
# set.seed(1)
# cv <- xgb.cv(params=params, data=dtrain, nrounds=3000, nfold=5,
#              early_stopping_rounds=50, verbose=0)
# best_n <- cv$best_iteration
# cat("best nrounds:", best_n, "\n")

# fin.mod <- xgb.train(params=params, data=dtrain, nrounds=best_n, verbose=0)

# pred.label <- predict(fin.mod, dtest)
# write.csv(data.frame(RowIndex = seq_along(pred.label), Prediction = pred.label),
#           "RegressionPredictLabel.csv", row.names = FALSE)

In [12]:
# best_n <- 410   # 用你xgb.cv选出来的

# params <- list(objective="reg:squarederror",
#                eta=0.03, max_depth=3, min_child_weight=5,
#                subsample=0.7, colsample_bytree=0.7, lambda=2)

# preds <- sapply(1:10, function(s){
#   set.seed(s)
#   m <- xgb.train(params=params, data=dtrain, nrounds=best_n, verbose=0)
#   predict(m, dtest)
# })
# pred.label <- rowMeans(preds)   # 10个seed平均
# write.csv(data.frame(RowIndex=seq_along(pred.label), Prediction=pred.label),
#           "RegressionPredictLabel1.csv", row.names=FALSE)

In [49]:
# 4.42,目前最优
params_B <- list(objective="reg:squarederror",
                 eta=0.05, max_depth=6,
                 subsample=0.8, colsample_bytree=0.8, min_child_weight=1)
set.seed(1)
cv <- xgb.cv(params=params_B, data=dtrain, nrounds=2000, nfold=5,
             early_stopping_rounds=50, verbose=0)
m <- xgb.train(params=params_B, data=dtrain, nrounds=cv$best_iteration, verbose=0)
pred.label <- predict(m, dtest)
write.csv(data.frame(RowIndex=seq_along(pred.label), Prediction=pred.label),
          "RegressionPredictLabel2.csv", row.names=FALSE)

In [50]:
cat("实际用的nrounds:", cv$best_iteration, "\n")

实际用的nrounds: 208 


In [45]:
# # 基于最优的调整测试
# # 5.05
# params_test <- list(objective="reg:squarederror",
#                  eta=0.05, max_depth=5,
#                  subsample=0.8, colsample_bytree=0.9, min_child_weight=1)
# set.seed(1)
# cv <- xgb.cv(params=params_test, data=dtrain, nrounds=2000, nfold=5,
#              early_stopping_rounds=50, verbose=0)
# m <- xgb.train(params=params_test, data=dtrain, nrounds=cv$best_iteration, verbose=0)
# pred.label <- predict(m, dtest)
# write.csv(data.frame(RowIndex=seq_along(pred.label), Prediction=pred.label),
#           "RegressionPredictLabel_test1.csv", row.names=FALSE)

In [47]:
# #4.56
# base <- list(objective="reg:squarederror", eta=0.05, max_depth=6,
#              subsample=0.8, colsample_bytree=0.8, min_child_weight=1)
# configs <- expand.grid(depth=c(6,7,8), seed=1:4)   # 用6及以上(容量够),12个模型
# preds <- sapply(1:nrow(configs), function(i){
#   set.seed(configs$seed[i])
#   p <- modifyList(base, list(max_depth=configs$depth[i]))
#   cv <- xgb.cv(params=p, data=dtrain, nrounds=2000, nfold=5, early_stopping_rounds=50, verbose=0)
#   xgb.train(params=p, data=dtrain, nrounds=cv$best_iteration, verbose=0) |> predict(dtest)
# })
# pred.label <- rowMeans(preds)
# write.csv(data.frame(RowIndex=seq_along(pred.label), Prediction=pred.label),
#           "RegressionPredictLabel_ensemble.csv", row.names=FALSE)

In [53]:
#400：4.21862， 800: 4.19993,  1200:4.19965
#新最佳，800/1200 sub_nr800/1200
params <- list(objective="reg:squarederror", eta=0.05, max_depth=6,
               subsample=0.8, colsample_bytree=0.8, min_child_weight=1)
for(nr in c(400, 800, 1200)){
  set.seed(1)
  m <- xgb.train(params=params, data=dtrain, nrounds=nr, verbose=0)
  p <- predict(m, dtest)
  write.csv(data.frame(RowIndex=seq_along(p), Prediction=p),
            paste0("sub_nr", nr, ".csv"), row.names=FALSE)
}

In [54]:
# for(d in c(4, 5, 7)){
#   params <- list(objective="reg:squarederror", eta=0.05, max_depth=d,
#                  subsample=0.8, colsample_bytree=0.8, min_child_weight=1)
#   set.seed(1)
#   m <- xgb.train(params=params, data=dtrain, nrounds=800, verbose=0)
#   p <- predict(m, dtest)
#   write.csv(data.frame(RowIndex=seq_along(p), Prediction=p),
#             paste0("sub_d", d, "_nr800.csv"), row.names=FALSE)
# }

In [55]:
p1 <- list(objective="reg:squarederror", eta=0.03, max_depth=6,
           subsample=0.8, colsample_bytree=0.8, min_child_weight=1)
m <- xgb.train(params=p1, data=dtrain, nrounds=1500, verbose=0)
write.csv(data.frame(RowIndex=1:90, Prediction=predict(m,dtest)), "sub_eta03.csv", row.names=FALSE)
